# TB-CXR: Severity-Stratified Tuberculosis Classification for USCIS Immigration Medical Examination
## DenseNet-121 · ResNet-50 · EfficientNet-B4 | Asymmetric Triage Loss | GradCAM | MC Dropout | CD-CTEI

**Author:** Azizur Rahman  
**Affiliation:** Indiana Wesleyan University · RadTH Technologies  
**Paper:** *Severity-Stratified Tuberculosis Classification from Chest Radiographs Aligned with CDC/USCIS Immigration Medical Examination Guidelines*  
**Target Venue:** Journal of Biomedical Informatics

---
### Notebook Contents
| # | Section |
|---|--------|
| 1 | Environment Setup |
| 2 | Dataset Loading & 4-Class USCIS Label Mapping |
| 3 | Data Augmentation & DataLoader |
| 4 | Model Architecture (DenseNet-121, ResNet-50, EfficientNet-B4) |
| 5 | Asymmetric Triage Loss |
| 6 | Training with Transfer Learning |
| 7 | ATL Ablation Study |
| 8 | Cross-Dataset Evaluation (Montgomery + Shenzhen) |
| 9 | ROC-AUC Curves |
| 10 | GradCAM Explainability |
| 11 | Monte Carlo Dropout Uncertainty |
| 12 | Civil Surgeon Screening Simulation |
| 13 | Cross-Demographic Equity (CD-CTEI) |
| 14 | Results Summary & Paper Tables |

---
### 4-Class CDC/USCIS Severity Mapping — Core Contribution
| Class | Label | X-Ray Finding | USCIS Outcome | Clinical Action |
|---|---|---|---|---|
| 0 | Normal | Clear lungs | **Cleared** | Routine immigration processing |
| 1 | Inactive TB | Calcified nodules, fibrous scars | **Class B2** | Cleared + 30-day follow-up |
| 2 | Active TB | Infiltrates, consolidation | **Class B1** | Sputum smear required |
| 3 | Severe TB | Cavitation, miliary pattern | **Class A** | Not cleared — treatment first |

## 1. Environment Setup

In [ ]:
!pip install torch torchvision timm
!pip install scikit-learn pandas numpy matplotlib seaborn tqdm
!pip install grad-cam opencv-python pillow
!pip install kaggle

In [ ]:
import os, random, warnings, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms, models
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score,
    confusion_matrix, recall_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NUM_CLASSES  = 4
IMG_SIZE     = 224
BATCH_SIZE   = 32
LABEL_NAMES  = ['Normal', 'Inactive_TB', 'Active_TB', 'Severe_TB']
USCIS_LABELS = ['Cleared', 'Class B2', 'Class B1', 'Class A']
USCIS_ACTIONS = [
    'Routine clearance',
    'Cleared + 30-day follow-up',
    'Sputum smear + further evaluation',
    'NOT cleared — treatment required'
]

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'\n4-Class USCIS Mapping:')
for i in range(NUM_CLASSES):
    print(f'  Class {i} | {LABEL_NAMES[i]:<15} → {USCIS_LABELS[i]}: {USCIS_ACTIONS[i]}')

## 2. Dataset Loading & 4-Class USCIS Label Mapping

In [ ]:
SEVERE_KEYWORDS   = ['cavit','miliary','bilateral extensive','large effusion',
                     'extensive consolidation','disseminated','progressive']
ACTIVE_KEYWORDS   = ['infiltrat','consolidat','ground glass','opacity',
                     'airspace','active','pneumonic','lesion','exudate']
INACTIVE_KEYWORDS = ['calcif','fibro','scar','healed','old','chronic',
                     'nodule','granuloma','pleural thicken','residual']

def classify_severity(report: str, has_tb: bool) -> int:
    if not has_tb: return 0
    if not isinstance(report, str) or not report.strip(): return 2
    text = report.lower()
    if any(k in text for k in SEVERE_KEYWORDS):   return 3
    if any(k in text for k in ACTIVE_KEYWORDS):   return 2
    if any(k in text for k in INACTIVE_KEYWORDS): return 1
    return 2


def load_dataset(data_dir: str, dataset_name: str = 'dataset') -> pd.DataFrame:
    rows = []
    for folder, has_tb in [('Normal', False), ('TB-Negative', False),
                            ('TB', True), ('TB-Positive', True)]:
        fp = os.path.join(data_dir, folder)
        if not os.path.exists(fp): continue
        report_dir = os.path.join(data_dir, 'ClinicalReadings')
        for fname in os.listdir(fp):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            report = ''
            rpath  = os.path.join(report_dir, fname.replace('.png', '.txt').replace('.jpg', '.txt'))
            if os.path.exists(rpath):
                with open(rpath, 'r', errors='ignore') as f: report = f.read()
            label = classify_severity(report, has_tb)
            rows.append({'image_path': os.path.join(fp, fname),
                         'label': label, 'label_name': LABEL_NAMES[label],
                         'uscis': USCIS_LABELS[label], 'dataset': dataset_name})
    if not rows:
        for fname in os.listdir(data_dir):
            if not fname.lower().endswith(('.png', '.jpg')): continue
            has_tb = fname.split('_')[-1].split('.')[0] == '1'
            label  = classify_severity('', has_tb)
            rows.append({'image_path': os.path.join(data_dir, fname),
                         'label': label, 'label_name': LABEL_NAMES[label],
                         'uscis': USCIS_LABELS[label], 'dataset': dataset_name})
    df = pd.DataFrame(rows)
    print(f'\n{dataset_name}: {len(df)} images')
    print(df['label_name'].value_counts().to_string())
    return df


DATA_ROOT = 'data'
# Uncomment after downloading:
# df_tbx   = load_dataset(f'{DATA_ROOT}/tbx11k',     'TBX11K')
# df_mont  = load_dataset(f'{DATA_ROOT}/montgomery',  'Montgomery')
# df_shen  = load_dataset(f'{DATA_ROOT}/shenzhen',    'Shenzhen')
# train_df, temp = train_test_split(df_tbx, test_size=0.3, stratify=df_tbx['label'], random_state=SEED)
# val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp['label'], random_state=SEED)
print('Label functions ready. Update DATA_ROOT and uncomment after downloading.')

## 3. Data Augmentation & DataLoader

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class TBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.long)

def make_loaders(train_df, val_df, test_df):
    counts  = train_df['label'].value_counts().sort_index().values
    weights = 1.0 / counts
    sw      = weights[train_df['label'].values]
    sampler = WeightedRandomSampler(sw, len(sw))
    train_ldr = DataLoader(TBDataset(train_df, train_transforms), BATCH_SIZE, sampler=sampler, num_workers=2)
    val_ldr   = DataLoader(TBDataset(val_df,   val_transforms),   BATCH_SIZE, shuffle=False, num_workers=2)
    test_ldr  = DataLoader(TBDataset(test_df,  val_transforms),   BATCH_SIZE, shuffle=False, num_workers=2)
    return train_ldr, val_ldr, test_ldr

print('DataLoader ready.')

## 4. Model Architecture — DenseNet-121 · ResNet-50 · EfficientNet-B4

In [ ]:
def build_densenet121(num_classes=NUM_CLASSES, dropout=0.5):
    m = models.densenet121(weights='DEFAULT')
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.classifier.in_features, num_classes))
    return m

def build_resnet50(num_classes=NUM_CLASSES, dropout=0.5):
    m = models.resnet50(weights='DEFAULT')
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    return m

def build_efficientnet_b4(num_classes=NUM_CLASSES, dropout=0.4):
    m = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
    m.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(m.num_features, 256),
        nn.ReLU(),
        nn.Dropout(dropout / 2),
        nn.Linear(256, num_classes)
    )
    return m

dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
for name, model in [('DenseNet-121', build_densenet121()),
                    ('ResNet-50',    build_resnet50()),
                    ('EfficientNet-B4', build_efficientnet_b4())]:
    n = sum(p.numel() for p in model.parameters())
    print(f'{name:<18} params={n:>10,}  output={model(dummy).shape}')

## 5. Asymmetric Triage Loss
Missing Severe TB (Class A) is the worst outcome: patient enters immigration stream with active disease.
We assign 5× higher penalty for Severe TB misclassification.

In [ ]:
class AsymmetricTriageLoss(nn.Module):
    """
    Weighted cross-entropy with CDC-aligned clinical cost weights.
    Normal=1.0  Inactive=2.0  Active=3.0  Severe=5.0
    """
    def __init__(self, weights=None, device=DEVICE):
        super().__init__()
        if weights is None:
            weights = torch.tensor([1.0, 2.0, 3.0, 5.0], dtype=torch.float32)
        self.ce = nn.CrossEntropyLoss(weight=weights.to(device))
    def forward(self, logits, targets): return self.ce(logits, targets)

print('ATL weights:')
for i, (n, w) in enumerate(zip(LABEL_NAMES, [1.0, 2.0, 3.0, 5.0])):
    print(f'  Class {i} {n:<15} weight={w}  USCIS: {USCIS_LABELS[i]}')

## 6. Training with Transfer Learning

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, preds_all, trues_all = 0, [], []
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds_all.extend(out.detach().argmax(1).cpu().numpy())
        trues_all.extend(labels.cpu().numpy())
    scheduler.step()
    return total_loss/len(loader), f1_score(trues_all, preds_all, average='macro', zero_division=0)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, trues_all = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        total_loss += criterion(out, labels).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        trues_all.extend(labels.cpu().numpy())
    return (f1_score(trues_all, preds_all, average='macro', zero_division=0),
            total_loss/len(loader), preds_all, trues_all)

def run_training(model, train_ldr, val_ldr, model_name, epochs=20, lr=2e-4, ckpt_dir='checkpoints'):
    ckpt_path = os.path.join(ckpt_dir, model_name, 'best.pt')
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    if os.path.exists(ckpt_path):
        print(f'Loading checkpoint: {model_name}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        return model, {}
    model     = model.to(DEVICE)
    criterion = AsymmetricTriageLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    best_f1, best_state = 0, None
    history = {'train_loss':[],'val_loss':[],'train_f1':[],'val_f1':[]}
    print(f'\nTraining {model_name} — {epochs} epochs')
    print('='*70)
    for epoch in range(1, epochs+1):
        tl, tf = train_epoch(model, train_ldr, optimizer, scheduler, criterion, DEVICE)
        vf, vl, _, _ = evaluate(model, val_ldr, criterion, DEVICE)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_f1'].append(tf);  history['val_f1'].append(vf)
        print(f'Epoch {epoch:>2}/{epochs}  train_loss={tl:.4f}  train_F1={tf:.4f}  val_loss={vl:.4f}  val_F1={vf:.4f}')
        if vf > best_f1:
            best_f1 = vf
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt_path)
    print(f'Best val F1: {best_f1:.4f}  →  {ckpt_path}')
    return model, history

print('Training ready.')
print('  dn121, h1 = run_training(build_densenet121(), train_ldr, val_ldr, "densenet121")')
print('  rn50,  h2 = run_training(build_resnet50(),   train_ldr, val_ldr, "resnet50")')
print('  eb4,   h3 = run_training(build_efficientnet_b4(), train_ldr, val_ldr, "efficientnet_b4")')

## 7. ATL Ablation Study
Compare Asymmetric Triage Loss vs standard Cross-Entropy vs Focal Loss.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, device=DEVICE):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

@torch.no_grad()
def evaluate_with_loss(model, loader, loss_fn, device):
    model.eval()
    preds_all, trues_all = [], []
    for imgs, labels in loader:
        out = model(imgs.to(device))
        preds_all.extend(out.argmax(1).cpu().numpy())
        trues_all.extend(labels.numpy())
    severe_recall = recall_score(
        [1 if t==3 else 0 for t in trues_all],
        [1 if p==3 else 0 for p in preds_all], zero_division=0)
    return f1_score(trues_all, preds_all, average='macro', zero_division=0), severe_recall

def run_ablation(train_ldr, val_ldr, test_ldr, epochs=10):
    loss_fns = {
        'Cross-Entropy': nn.CrossEntropyLoss(),
        'Focal (γ=2)':   FocalLoss(gamma=2.0),
        'ATL (Ours)':    AsymmetricTriageLoss()
    }
    results = {}
    for name, loss_fn in loss_fns.items():
        print(f'\nAblation: {name}')
        model = build_densenet121().to(DEVICE)
        opt   = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
        for epoch in range(epochs):
            model.train()
            for imgs, labels in train_ldr:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opt.zero_grad()
                loss_fn(model(imgs), labels).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sch.step()
        f1, sr = evaluate_with_loss(model, test_ldr, loss_fn, DEVICE)
        results[name] = {'macro_f1': f1, 'severe_recall': sr}
        print(f'  Macro F1={f1:.4f}  Severe TB Recall={sr:.4f}')

    print('\nAblation Summary:')
    print(f'{"Loss":<20} {"Macro F1":>10} {"Severe Recall":>14}')
    print('-'*46)
    for k, v in results.items():
        print(f'{k:<20} {v["macro_f1"]:>10.4f} {v["severe_recall"]:>14.4f}')
    return results

print('ATL ablation ready.  Call: ablation_results = run_ablation(train_ldr, val_ldr, test_ldr)')

## 8. Cross-Dataset Evaluation — Montgomery & Shenzhen
Train on TBX11K, evaluate zero-shot on hospitals from different countries.

In [ ]:
@torch.no_grad()
def evaluate_detailed(model, loader, device, dataset_name=''):
    model.eval()
    preds_all, trues_all, probs_all = [], [], []
    for imgs, labels in loader:
        out = model(imgs.to(device))
        probs_all.extend(F.softmax(out, dim=1).cpu().numpy())
        preds_all.extend(out.argmax(1).cpu().numpy())
        trues_all.extend(labels.numpy())
    macro_f1 = f1_score(trues_all, preds_all, average='macro', zero_division=0)
    per_class = f1_score(trues_all, preds_all, average=None, zero_division=0)
    severe_recall = recall_score(
        [1 if t==3 else 0 for t in trues_all],
        [1 if p==3 else 0 for p in preds_all], zero_division=0)
    print(f'\n{"="*60}')
    print(f'Dataset: {dataset_name}  |  Macro F1: {macro_f1:.4f}')
    print(f'Severe TB Recall (Class A): {severe_recall:.4f}')
    print(classification_report(trues_all, preds_all, target_names=LABEL_NAMES, zero_division=0))
    return {'macro_f1': macro_f1, 'per_class': per_class,
            'severe_recall': severe_recall, 'preds': preds_all,
            'trues': trues_all, 'probs': np.array(probs_all)}

def plot_confusion_matrix(results, model_name):
    cm = confusion_matrix(results['trues'], results['preds'])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    short = ['Normal','Inactive','Active','Severe']
    for ax, data, fmt, title in zip([ax1, ax2],
                                     [cm, cm.astype(float)/cm.sum(axis=1,keepdims=True)],
                                     ['d', '.1%'],
                                     ['Count', 'Normalized']):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                    xticklabels=short, yticklabels=short)
        ax.set(xlabel='Predicted', ylabel='True',
               title=f'{model_name} Confusion Matrix ({title})')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Cross-dataset evaluation ready.')

## 9. ROC-AUC Curves — Per Class (One-vs-Rest)

In [ ]:
def plot_roc_curves(results_dict: dict):
    """
    results_dict: {'DenseNet-121': results, 'ResNet-50': results, 'EfficientNet-B4': results}
    Each results dict must contain 'trues' and 'probs' (softmax probabilities).
    """
    classes = LABEL_NAMES
    n_cls   = len(classes)
    colors  = ['#2980B9', '#E74C3C', '#2ECC71', '#F39C12']
    model_styles = ['-', '--', ':']

    fig, axes = plt.subplots(1, n_cls, figsize=(5*n_cls, 5))
    for c, (cls_name, ax) in enumerate(zip(classes, axes)):
        for (model_name, results), ls in zip(results_dict.items(), model_styles):
            trues_bin = [1 if t == c else 0 for t in results['trues']]
            probs_cls = np.array(results['probs'])[:, c]
            fpr, tpr, _ = roc_curve(trues_bin, probs_cls)
            roc_auc     = auc(fpr, tpr)
            ax.plot(fpr, tpr, linestyle=ls, linewidth=2,
                    label=f'{model_name} (AUC={roc_auc:.3f})')
        ax.plot([0,1],[0,1],'k--', linewidth=1, alpha=0.5)
        ax.set(xlabel='FPR', ylabel='TPR',
               title=f'{cls_name}\n(USCIS: {USCIS_LABELS[c]})')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        if c == 3:  # Severe TB — add clinical note
            ax.text(0.35, 0.05, 'HIGH PRIORITY\nClass A = Not Cleared',
                    fontsize=8, color='red', alpha=0.7,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    plt.suptitle('Per-Class ROC Curves — TB Severity (One-vs-Rest)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def print_auc_table(results_dict: dict):
    print(f'\n{"Model":<20}', end='')
    for c in LABEL_NAMES: print(f'{c:>14}', end='')
    print(f'{"Macro AUC":>12}')
    print('-' * 80)
    for model_name, results in results_dict.items():
        trues_bin = label_binarize(results['trues'], classes=list(range(NUM_CLASSES)))
        aucs = []
        print(f'{model_name:<20}', end='')
        for c in range(NUM_CLASSES):
            a = roc_auc_score(trues_bin[:, c], np.array(results['probs'])[:, c])
            aucs.append(a)
            print(f'{a:>14.4f}', end='')
        print(f'{np.mean(aucs):>12.4f}')

print('ROC-AUC analysis ready.')
print('Call: plot_roc_curves({"DenseNet-121": dn_results, "ResNet-50": rn_results, "EfficientNet-B4": eb_results})')

## 10. GradCAM Explainability
Highlights lung regions driving each USCIS severity classification.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        target_layer.register_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))

    def generate(self, img_tensor, class_idx=None):
        self.model.eval()
        img = img_tensor.unsqueeze(0).to(DEVICE)
        out = self.model(img)
        if class_idx is None: class_idx = out.argmax().item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.grads.mean(dim=[2,3], keepdim=True)
        cam = F.relu((weights * self.acts).sum(1, keepdim=True))
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam, class_idx


def visualize_gradcam(model, image_paths, true_labels=None):
    target = model.features.denseblock4.denselayer16.conv2
    gcam   = GradCAM(model, target)
    n      = len(image_paths)
    inv    = transforms.Normalize(
        mean=[-0.485/0.229,-0.456/0.224,-0.406/0.225],
        std=[1/0.229,1/0.224,1/0.225])
    fig, axes = plt.subplots(2, n, figsize=(4*n, 9))
    if n == 1: axes = axes.reshape(2, 1)
    for i, path in enumerate(image_paths):
        img_raw = Image.open(path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
        img_t   = val_transforms(img_raw)
        cam, pred = gcam.generate(img_t)
        img_np  = np.array(inv(img_t).permute(1,2,0).clamp(0,1))
        heatmap = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cam),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)/255.0
        overlay = np.clip(0.6*img_np + 0.4*heatmap, 0, 1)
        gt = LABEL_NAMES[true_labels[i]] if true_labels else ''
        axes[0,i].imshow(img_np); axes[0,i].set_title(f'GT: {gt}', fontsize=10)
        axes[1,i].imshow(overlay)
        color = 'red' if pred == 3 else 'orange' if pred == 2 else 'black'
        axes[1,i].set_title(f'{LABEL_NAMES[pred]}\n{USCIS_LABELS[pred]}', fontsize=9, color=color)
        for ax in [axes[0,i],axes[1,i]]: ax.axis('off')
    plt.suptitle('GradCAM — TB Severity Classification\nRed = lung regions driving USCIS classification decision',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_tb.png', dpi=150, bbox_inches='tight')
    plt.show()

print('GradCAM ready.')

## 11. Monte Carlo Dropout Uncertainty

In [ ]:
def mc_predict(model, img_tensor, n_samples=30):
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train()
    img = img_tensor.unsqueeze(0).to(DEVICE)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            probs.append(F.softmax(model(img), dim=1).cpu().numpy())
    probs      = np.stack(probs).squeeze(1)
    mean_probs = probs.mean(0)
    pred       = mean_probs.argmax()
    confidence = mean_probs.max()
    entropy    = -np.sum(mean_probs * np.log(mean_probs + 1e-8))
    flag       = confidence < 0.75 or pred >= 2
    return {'prediction': int(pred), 'label': LABEL_NAMES[pred],
            'uscis': USCIS_LABELS[pred], 'confidence': float(confidence),
            'uncertainty': float(entropy), 'flag': flag,
            'mean_probs': mean_probs}

print('MC Dropout ready.')
print('Flag conditions: confidence < 0.75 OR predicted class >= Active TB')

## 12. Civil Surgeon Screening Simulation
Quantifies how much AI reduces civil surgeon workload while maintaining zero missed Class A cases.

In [ ]:
def civil_surgeon_simulation(model, test_df, confidence_threshold=0.75):
    """
    Simulate AI-assisted USCIS Form I-693 screening workflow.
    AI handles clear Normal/Inactive cases; civil surgeon reviews flagged cases.
    """
    auto_cleared, flagged = 0, 0
    missed_class_a = 0
    flag_breakdown = {'low_confidence': 0, 'active_tb': 0, 'severe_tb': 0}

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Simulating'):
        img = val_transforms(Image.open(row['image_path']).convert('RGB'))
        res = mc_predict(model, img)

        if res['flag']:
            flagged += 1
            if res['confidence'] < confidence_threshold:  flag_breakdown['low_confidence'] += 1
            if res['prediction'] == 2:                    flag_breakdown['active_tb'] += 1
            if res['prediction'] == 3:                    flag_breakdown['severe_tb'] += 1
        else:
            auto_cleared += 1
            if row['label'] == 3:  # Severe TB missed by AI
                missed_class_a += 1

    total = len(test_df)
    print(f'\nCivil Surgeon Screening Simulation')
    print(f'Total applicants:          {total}')
    print(f'AI auto-cleared (Normal/Inactive): {auto_cleared} ({100*auto_cleared/total:.1f}%)')
    print(f'Flagged for civil surgeon: {flagged} ({100*flagged/total:.1f}%)')
    print(f'Workload reduction:        {100*auto_cleared/total:.1f}%')
    print(f'Missed Class A (Severe TB): {missed_class_a}  (target: 0)')
    print(f'\nFlag breakdown:')
    for reason, count in flag_breakdown.items():
        print(f'  {reason}: {count}')

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
    ax1.pie([auto_cleared, flagged],
            labels=[f'AI auto-cleared\n({100*auto_cleared/total:.1f}%)',
                    f'Civil surgeon review\n({100*flagged/total:.1f}%)'],
            colors=['#2ECC71','#E74C3C'], autopct='%1.1f%%', startangle=90)
    ax1.set_title('USCIS Screening Workflow\nAI Workload Distribution')
    thresholds = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
    reductions = []
    for thresh in thresholds:
        cleared = sum(1 for _, r in test_df.iterrows()
                     if mc_predict(model, val_transforms(
                         Image.open(r['image_path']).convert('RGB')))['confidence'] >= thresh
                     and mc_predict(model, val_transforms(
                         Image.open(r['image_path']).convert('RGB')))['prediction'] < 2)
        reductions.append(100 * cleared / total)
    ax2.plot(thresholds, reductions, 'b-o', linewidth=2)
    ax2.axvline(confidence_threshold, color='red', linestyle='--', label=f'Selected threshold={confidence_threshold}')
    ax2.set(xlabel='Confidence Threshold', ylabel='Workload Reduction (%)',
            title='Threshold vs. Civil Surgeon Burden Reduction')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('screening_simulation.png', dpi=150, bbox_inches='tight')
    plt.show()
    return auto_cleared, flagged, missed_class_a

print('Screening simulation ready.')

## 13. Cross-Demographic Equity (CD-CTEI)

In [ ]:
def compute_cdctei(f1_dict: dict) -> float:
    vals = np.array(list(f1_dict.values()))
    mu, sigma = vals.mean(), vals.std()
    return float(1 - sigma/mu) if mu > 0 else 0.0


def equity_analysis(model, test_df, device, group_col='age_group'):
    if group_col not in test_df.columns:
        print(f'Add {group_col} column to test_df for equity analysis.'); return
    results = {}
    criterion = AsymmetricTriageLoss()
    for group in sorted(test_df[group_col].unique()):
        sub = test_df[test_df[group_col]==group].reset_index(drop=True)
        ldr = DataLoader(TBDataset(sub, val_transforms), BATCH_SIZE, shuffle=False)
        f1, _, _, _ = evaluate(model, ldr, criterion, device)
        results[str(group)] = f1
    ctei = compute_cdctei(results)
    print(f'\nCD-CTEI: {ctei:.4f}  (threshold >= 0.95)')
    print(f'{"Group":<20} {"Macro F1":>10}')
    print('-'*32)
    for g, f in sorted(results.items(), key=lambda x: x[1]):
        flag = '⚠' if f < 0.85 else '✓'
        print(f'{g:<20} {f:>10.4f}  {flag}')
    fig, ax = plt.subplots(figsize=(9, 4))
    groups = list(results.keys()); f1s = [results[g] for g in groups]
    ax.bar(groups, f1s, color=['#2980B9' if f>=0.85 else '#E74C3C' for f in f1s], alpha=0.85)
    ax.axhline(0.85, color='red', linestyle='--', label='Min threshold')
    ax.set(ylabel='Macro F1', title=f'CD-CTEI={ctei:.3f} — Per-Demographic Performance')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig('cdctei_equity.png', dpi=150, bbox_inches='tight')
    plt.show()
    return results, ctei

print('CD-CTEI equity analysis ready.')

## 14. Results Summary & Paper Tables

In [ ]:
def print_paper_tables(results_dict: dict, ext_results: dict = None):
    print('\nTable 2. Per-Class F1 — All Models')
    print('='*85)
    print(f'{"Class":<18}', end='')
    for name in results_dict: print(f'{name:>18}', end='')
    print(f'  USCIS Outcome')
    print('-'*85)
    for i, (label, uscis) in enumerate(zip(LABEL_NAMES, USCIS_LABELS)):
        print(f'{label:<18}', end='')
        for res in results_dict.values(): print(f'{res["per_class"][i]:>18.4f}', end='')
        print(f'  {uscis}')
    print('-'*85)
    print(f'{"Macro F1":<18}', end='')
    for res in results_dict.values(): print(f'{res["macro_f1"]:>18.4f}', end='')
    print()
    print(f'{"Severe Recall":<18}', end='')
    for res in results_dict.values(): print(f'{res["severe_recall"]:>18.4f}', end='')
    print()

    if ext_results:
        print('\nTable 3. Cross-Site Generalization')
        print('='*60)
        print(f'{"Dataset":<20}', end='')
        for name in results_dict: print(f'{name:>18}', end='')
        print()
        print('-'*60)
        for site, res_by_model in ext_results.items():
            print(f'{site:<20}', end='')
            for name in results_dict:
                print(f'{res_by_model.get(name, {}).get("macro_f1", 0):>18.4f}', end='')
            print()

print('Results functions ready.')

## Summary

| Component | Detail |
|---|---|
| **Task** | 4-class TB severity classification aligned with CDC/USCIS Form I-693 |
| **Models** | DenseNet-121 · ResNet-50 · EfficientNet-B4 |
| **Loss** | Asymmetric Triage Loss (Severe TB weight=5×) |
| **Training set** | TBX11K (11,200 images) |
| **External validation** | Montgomery (138 images) + Shenzhen (662 images) |
| **Explainability** | GradCAM — lung region localization per severity class |
| **Uncertainty** | MC Dropout — flags Active/Severe TB for civil surgeon review |
| **Ablation** | ATL vs Cross-Entropy vs Focal Loss |
| **ROC-AUC** | Per-class one-vs-rest curves for all three models |
| **Fairness** | CD-CTEI across age/sex demographic subgroups |
| **Deployment** | `inference.py` standalone script for civil surgeon workstation |
| **NIW Hook** | USCIS Form I-693 mandates this exam for every US immigration applicant |

### Output Files
- `training_curves.png`  
- `confusion_matrix_densenet121.png`  
- `roc_curves.png`  
- `gradcam_tb.png`  
- `screening_simulation.png`  
- `cdctei_equity.png`  
- `checkpoints/densenet121/best.pt`  
- `checkpoints/resnet50/best.pt`  
- `checkpoints/efficientnet_b4/best.pt`